# R Master v2 · 不再上传的手机版云跑

这一版**不需要再上传 Mona**。

它会直接从你上一次 v1 已经写入的 Google Drive 缓存读取：

`MyDrive/R_Master/cache/Mona_Source.blend`

并继续复用已经缓存的 Blender 4.4.3 压缩包。

v2 做的是：

- 保留 v1 已验证通过的骨段比例和髋宽预览；
- 在预览副本上清理 `Mona_Main` 的衣物 Mask / SurfaceDeform / Cloth 干扰；
- 降低 Multires/Subdivision 的预览级别，让云跑更快；
- 输出 3 张全身灰模 + 2 张骨盆近景；
- 重 `.blend` 留在 Drive，手机只下载很小的 review ZIP。

仍然不会修改原始 Mona、不会烘焙最终 Rest Pose、不会导出最终 VRM。


In [ ]:
from google.colab import drive, files
from IPython.display import display, Image, Markdown
from pathlib import Path
import os, shutil, subprocess, urllib.request, zipfile, json

BLENDER_VERSION = "4.4.3"
BLENDER_URL = "https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
SCRIPT_COMMIT = "96543711261178386da1ece924af0c56a260f934"
SCRIPT_URL = f"https://raw.githubusercontent.com/hexiangyu481-commits/-erdan-lab-mobile/{SCRIPT_COMMIT}/red-r-master/blender/R_Master_BuildPreview_v2.py"

LOCAL = Path("/content/r_master_v2")
LOCAL_OUT = LOCAL / "output"
LOCAL_ARCHIVE = LOCAL / f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
LOCAL_BLENDER_DIR = LOCAL / f"blender-{BLENDER_VERSION}-linux-x64"
LOCAL_SCRIPT = LOCAL / "R_Master_BuildPreview_v2.py"
LOCAL_SOURCE = LOCAL / "Mona_Source.blend"
LOCAL_REVIEW = LOCAL / "R_Master_v2_Review.zip"

LOCAL.mkdir(parents=True, exist_ok=True)

print("R Master v2 · Drive 直读版")
print("① 挂载 Google Drive…")
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/R_Master")
DRIVE_CACHE = DRIVE_ROOT / "cache"
DRIVE_V1 = DRIVE_ROOT / "v1" / "latest"
DRIVE_V2 = DRIVE_ROOT / "v2" / "latest"
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_V2.mkdir(parents=True, exist_ok=True)

DRIVE_SOURCE = DRIVE_CACHE / "Mona_Source.blend"
DRIVE_ARCHIVE = DRIVE_CACHE / LOCAL_ARCHIVE.name

if not DRIVE_SOURCE.exists() or DRIVE_SOURCE.stat().st_size < 50 * 1024 * 1024:
    fallback = DRIVE_V1 / "R_Master_Align_v1_PREVIEW.blend"
    if fallback.exists() and fallback.stat().st_size > 50 * 1024 * 1024:
        print("Drive 源缓存缺失，但发现 v1 完整预览文件；用它恢复缓存…")
        shutil.copy2(fallback, DRIVE_SOURCE)
    else:
        raise RuntimeError(
            "Drive 里没找到 Mona 缓存。请先成功跑一次 v1；v2 本身不再提供重复上传入口。"
        )

print(f"✓ 直接读取 Drive Mona 缓存：{DRIVE_SOURCE.stat().st_size/1024/1024:.1f} MiB")
print("② 复制到 Colab 本地 SSD…")
shutil.copy2(DRIVE_SOURCE, LOCAL_SOURCE)

print("③ 准备 Blender…")
if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "xvfb", "libgl1", "libx11-6",
         "libxi6", "libxrender1", "libxfixes3", "libxkbcommon0", "libsm6"],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
    )

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size > 100 * 1024 * 1024:
    print("✓ 使用 Drive 里的 Blender 缓存")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
else:
    print("Drive 里没有 Blender 缓存，补下载一次…")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-O", str(LOCAL_ARCHIVE), BLENDER_URL],
        check=True
    )
    shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

if LOCAL_BLENDER_DIR.exists():
    shutil.rmtree(LOCAL_BLENDER_DIR)
subprocess.run(["tar", "-xf", str(LOCAL_ARCHIVE), "-C", str(LOCAL)], check=True)
BLENDER = LOCAL_BLENDER_DIR / "blender"
if not BLENDER.exists():
    raise RuntimeError("Blender 解压失败。")
print("✓ Blender 就绪")

print("④ 获取锁定的 v2 构建脚本…")
urllib.request.urlretrieve(SCRIPT_URL, LOCAL_SCRIPT)
print("✓ v2 script commit:", SCRIPT_COMMIT)

if LOCAL_OUT.exists():
    shutil.rmtree(LOCAL_OUT)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

print("⑤ 云端 Blender 正在做 clean-body v2…")
log_path = LOCAL_OUT / "R_Master_v2_blender.log"
cmd = [
    "xvfb-run", "-a", str(BLENDER),
    "--background", str(LOCAL_SOURCE),
    "--python", str(LOCAL_SCRIPT),
    "--", "--out", str(LOCAL_OUT),
]

with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        log.write(line)
        if ("R Master v2" in line) or ("Error" in line) or ("Traceback" in line):
            print(line.rstrip())
    code = proc.wait()

if code != 0:
    tail = log_path.read_text(encoding="utf-8", errors="replace")[-9000:]
    print("\n--- Blender 日志末尾 ---\n" + tail)
    raise RuntimeError(f"v2 Blender 运行失败，退出码 {code}。截图这一屏给二蛋即可。")

report_path = LOCAL_OUT / "R_Master_v2_report.json"
preview_blend = LOCAL_OUT / "R_Master_Align_v2_PREVIEW.blend"
imgs = [
    LOCAL_OUT / "R_Master_v2_clean_body_front.png",
    LOCAL_OUT / "R_Master_v2_clean_body_side.png",
    LOCAL_OUT / "R_Master_v2_clean_body_three_quarter.png",
    LOCAL_OUT / "R_Master_v2_pelvis_front.png",
    LOCAL_OUT / "R_Master_v2_pelvis_side.png",
]
required = [report_path, preview_blend, log_path, *imgs]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise RuntimeError("v2 缺少输出：" + ", ".join(missing))

report = json.loads(report_path.read_text(encoding="utf-8"))
print("\n✓ R Master v2 BUILD_OK")
print("  机械比例验收：", report.get("acceptance", {}).get("mechanical_preview_pass"))
print("  body modifier 清理：", report.get("body_modifier_cleaning", {}).get("disabled"))
print("  Rest Pose 烘焙：", report.get("rest_pose_baked"))

print("\n⑥ 保存重文件到 Google Drive…")
for p in DRIVE_V2.iterdir():
    if p.is_file():
        p.unlink()
for p in LOCAL_OUT.iterdir():
    if p.is_file():
        shutil.copy2(p, DRIVE_V2 / p.name)
print("✓ MyDrive/R_Master/v2/latest/")

print("\n⑦ 预览")
labels = ["全身正面", "全身侧面", "全身 3/4", "骨盆正面近景", "骨盆侧面近景"]
for title, path in zip(labels, imgs):
    display(Markdown(f"### {title}"))
    display(Image(filename=str(path), width=500))

print("\n⑧ 生成轻量 review ZIP…")
if LOCAL_REVIEW.exists():
    LOCAL_REVIEW.unlink()
with zipfile.ZipFile(LOCAL_REVIEW, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in [*imgs, report_path, log_path]:
        z.write(p, arcname=p.name)

shutil.copy2(LOCAL_REVIEW, DRIVE_V2 / LOCAL_REVIEW.name)
print(f"✓ Review ZIP：{LOCAL_REVIEW.stat().st_size/1024/1024:.1f} MiB")
print("重 .blend 已留在 Drive，不下载到手机。")
files.download(str(LOCAL_REVIEW))

